## Wczytanie danych i bibliotek

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../data/Hotel Reservations.csv')

# 2. Przetwarzanie danych

### Usunięcie kolumny Booking_ID

In [3]:
df = df.drop('Booking_ID', axis = 1)
df.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled
3,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,2018,5,20,Online,0,0,0,100.00,0,Canceled
4,2,0,1,1,Not Selected,0,Room_Type 1,48,2018,4,11,Online,0,0,0,94.50,0,Canceled


Zmienna ta nie wnosiłą nic wartościowego pod względem implementacji modelu.

### Kodowanie zmiennej docelowej

In [4]:
target = {'Canceled': 1, 'Not_Canceled': 0} # Zmiana booking_status na liczbe
df['booking_status'] = df['booking_status'].replace(target)
print("Canceled = 1, Not_Canceled = 0")
df.describe()

Canceled = 1, Not_Canceled = 0


,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,arrival_year,arrival_month,arrival_date,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
count,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000,36275.000000
mean,1.844962,0.105279,0.810724,2.204300,0.030986,85.232557,2017.820427,7.423653,15.596995,0.025637,0.023349,0.153411,103.423539,0.619655,0.327636
std,0.518715,0.402648,0.870644,1.410905,0.173281,85.930817,0.383836,3.069894,8.740447,0.158053,0.368331,1.754171,35.089424,0.786236,0.469358
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2017.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.000000,0.000000,1.000000,0.000000,17.000000,2018.000000,5.000000,8.000000,0.000000,0.000000,0.000000,80.300000,0.000000,0.000000
50%,2.000000,0.000000,1.000000,2.000000,0.000000,57.000000,2018.000000,8.000000,16.000000,0.000000,0.000000,0.000000,99.450000,0.000000,0.000000
75%,2.000000,0.000000,2.000000,3.000000,0.000000,126.000000,2018.000000,10.000000,23.000000,0.000000,0.000000,0.000000,120.000000,1.000000,1.000000
max,4.000000,10.000000,7.000000,17.000000,1.000000,443.000000,2018.000000,12.000000,31.000000,1.000000,13.000000,58.000000,540.000000,5.000000,1.000000


### Dodanie nowych kolumn

In [5]:
df['is_free_room'] = (df['avg_price_per_room'] == 0).astype(int)

W EDA wyszło, że dla ceny pokoju wynoszącej 0, bardzo rzadko rezerwacja jest anulowana. Z tego względu tworzona jest kolumna `is_free_room`, która binarnie mówi o tym, czy pokój był darmowy.

In [6]:
total_nights = df['no_of_week_nights'] + df['no_of_weekend_nights']

df['is_zero_nights'] = (total_nights == 0).astype(int)

zero_nights_count = df['is_zero_nights'].sum()
print(f"Liczba rezerwacji z liczbą nocy równą 0: {zero_nights_count}")

zero_nights_rates = df.groupby('is_zero_nights')['booking_status'].mean() * 100

print("\nWskaźnik anulacji (%):")
print(f"Standardowy pobyt (więcej niż 0 nocy): {zero_nights_rates[0]:.2f}%")
if zero_nights_count > 0:
    print(f"Pobyt 0-dniowy (is_zero_nights = 1):   {zero_nights_rates[1]:.2f}%")

Liczba rezerwacji z liczbą nocy równą 0: 78

Wskaźnik anulacji (%):
Standardowy pobyt (więcej niż 0 nocy): 32.83%
Pobyt 0-dniowy (is_zero_nights = 1):   2.56%


In [7]:

print("Ilu gości spełnia oba warunki na raz?")
crosstab_result = pd.crosstab(
    df['is_zero_nights'], 
    df['is_free_room'], 
    margins=True,
    rownames=['Day Use (0 nocy)'], 
    colnames=['Darmowy (0 zł)']
)
display(crosstab_result)

korelacja = df[['is_zero_nights', 'is_free_room']].corr().iloc[0, 1]
print(f"\nWspółczynnik korelacji wynosi: {korelacja:.4f}")

Ilu gości spełnia oba warunki na raz?


Darmowy (0 zł),0,1,All
Day Use (0 nocy),,,
0,35730,467,36197
1,0,78,78
All,35730,545,36275



Współczynnik korelacji wynosi: 0.3759


### Podział danych na zbiór treningowy, walidacyjny i testowy w proporcjach 80/10/10

In [8]:
X = df.drop('booking_status', axis = 1)
y = df['booking_status']

test_size = 0.10
val_size = 0.10
X_train_val, X_test, y_train_val, y_test = train_test_split( # 10%
    X, y, test_size=test_size, stratify=y, random_state=123)

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_size/(1 - test_size), stratify=y_train_val, random_state=123)

### Kodowanie zmiennych kategorycznych

In [10]:
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeryczne cechy:", num_features)
print("Kategoryczne cechy:", cat_features)

Numeryczne cechy: ['no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights', 'required_car_parking_space', 'lead_time', 'arrival_year', 'arrival_month', 'arrival_date', 'repeated_guest', 'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled', 'avg_price_per_room', 'no_of_special_requests', 'is_free_room', 'is_zero_nights']
Kategoryczne cechy: ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type']


In [11]:
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

In [12]:
data_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

data_pipeline.fit(X_train)

feature_names = (
    pd.Index(data_pipeline.named_steps['preprocessor'].get_feature_names_out())
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

X_train_processed = pd.DataFrame(data_pipeline.transform(X_train),
                                columns=feature_names)

X_val_processed = pd.DataFrame(data_pipeline.transform(X_val),
                                columns=feature_names)

X_test_processed = pd.DataFrame(data_pipeline.transform(X_test),
                                columns=feature_names)

### Zapisanie przetworzonych danych

In [13]:
import joblib, os

joblib.dump({
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    'X_train_processed': X_train_processed,
    'X_val_processed':   X_val_processed,
    'X_test_processed':  X_test_processed,
    'feature_names': feature_names,
    'df': df,
}, '../data/processed_data.pkl')

['../data/processed_data.pkl']